# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*
## 1) My Lane (or Freestyle) and Why

### Lane Selected: **Refresh / Content Opportunity Scoring** (Core Lane 2)

**Why this lane?**

1. **The starter dataset is built for it.** The repo ships `data/raw/content_refresh_anonymized.csv` (~30,000 rows), and the entire starter pipeline (`scripts/01`–`05`) is already a working end-to-end example of this exact problem. This means I can iterate from a verified baseline instead of starting from zero.

2. **The decision is real and constrained.** Content teams have limited review capacity. They cannot refresh every page. The question "which pages should we review first?" is a prioritization problem with a clear action (review → refresh/expand/protect/prune/monitor) and a measurable cost of getting it wrong.

3. **The baseline is beatable, but not trivially.** The starter results show baseline rules achieve Precision@50 = 0.240 (about 12 correct out of top 50), while a random forest reaches 0.740 (about 37 correct). That is a ~3× improvement in reviewer productivity — but it is earned on a 30K-row slice, not the full 79M-row warehouse. There is room to build a stronger label, a better feature window, and more honest validation on the warehouse data.

4. **I am not interested in "train a model" as an end goal.** I am interested in whether a ranked queue with reason codes can help a human reviewer spend their limited time on pages where evidence of decline or opportunity is strongest. The model is a tool; the decision support is the deliverable.



# Week 1: Research Question & Problem Framing

## 1) My Lane and Why
I am choosing **Lane 1: Content Prioritization (Ranking/Scoring)** to identify which content items are at risk of traffic decline and need immediate editorial review.

Instead of treating this as a simple classification problem ("will it decline?"), treating it as a ranking problem directly serves the business decision of where to allocate limited editor hours. By identifying the top $K$ items most vulnerable to performance drops, we optimize resource deployment.

Our exploration of the starter dataset below provides the baseline numbers that justify why this lane is both viable and valuable for the next 7 weeks.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

> **Which content pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring — based on observable search and engagement signals from the prior 90 days?**

### Unit of Analysis
One row = **one content page** (deduplicated by `content_id`). The grain is page-level, not daily. Features are aggregated over a 90-day window; the target is a future-window outcome (e.g., decline over the next 30 days) or a proxy label derived from the current window.

### The Decision This Improves
A content strategist or SEO reviewer decides **which pages to spend their limited time on today.** Without a ranked queue, they either:
- Review pages in random order (inefficient), or
- Use simple rules like "stale and visible" (better, but misses nuance and interaction effects).

A scored queue lets them start with the pages where the evidence of opportunity or risk is strongest.

### The Action Someone Takes
From the ranked output, a reviewer inspects the top-K pages and chooses one of:
- **Refresh** — update outdated facts, stats, or examples;
- **Expand** — add depth where the page is thin relative to demand;
- **Protect** — monitor a high-performing page showing early decay signals;
- **Prune** — consider consolidation if the page is redundant or cannibalizing another;
- **Monitor** — no action now, but flag for re-check in the next cycle.

Each page carries **reason codes** (e.g., `declining_with_demand`, `stale_visible_page`, `low_ctr_visible_page`) so the reviewer understands *why* it surfaced.

### Cost of a Wrong Recommendation

| Type | What happens | Cost |
|------|-------------|------|
| **False Positive** | We flag a page for review, but it does not need changes. | Reviewer time is wasted. Opportunity cost: another page that *did* need help was pushed down the queue. |
| **False Negative** | We miss a page that is genuinely declining. | Traffic and revenue loss continues until the decline is noticed manually. For high-impression pages, this can be significant. |
| **Wrong Action** | We recommend "refresh" but the page actually needs "prune/merge." | Effort is spent editing a page that should have been consolidated. May create duplicate content issues. |

The baseline wastes ~38 out of every 50 review slots (P@50 = 0.240). Improving that to even 0.500 means 25 more correct pages reviewed per batch. At scale, that is a massive efficiency gain.

### Why Data / ML Can Help Here
- **Signal is noisy and high-dimensional.** A page's "need" depends on impressions, clicks, CTR, position, age, freshness, engagement, scroll rate, word count, content type, intent, and trend direction — plus their interactions. A human cannot hold all of this in working memory for 500,000 pages.
- **Patterns exist but are not obvious.** The random forest beating the baseline by 3× on the starter slice suggests that interaction effects (e.g., "old + high impressions + declining CTR") carry signal that simple rules miss.
- **The decision is repetitive.** The same prioritization happens every week. A model that is validated, monitored, and updated is more consistent than ad-hoc human judgment at scale.
- **The output is decision-support, not automation.** The model ranks; the human decides. This keeps the human in the loop and avoids overclaiming.

In [ ]:
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# 1. Total records and unique clients
total_rows = len(df)
unique_clients = df['client_id'].nunique()

# 2. Measure the actual baseline of the target (declining pages)
# Note: 'is_declining_label' is our observed outcome trap warning check
declining_count = df['is_declining_label'].sum()
decline_rate = (declining_count / total_rows) * 100

# 3. Check missingness or anomalies (e.g., avg_position == 0 means "no data")
no_rank_data = (df['avg_position'] == 0).sum()

print(f"--- Starter Dataset Insights ---")
print(f"Total Content Items (Rows): {total_rows:,}")
print(f"Unique Clients: {unique_clients}")
print(f"Declining Items Count: {declining_count:,} ({decline_rate:.2f}%)")
print(f"Items with No Rank Data (avg_position == 0): {no_rank_data:,}")

FileNotFoundError: [Errno 2] No such file or directory: '../../data/raw/content_refresh_anonymized.csv'

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.